In [ ]:
import pandas as pd
import numpy as np
import glob
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Load Bitcoin data
files    = glob.glob("./data/*.csv")
btc_file = [f for f in files if 'Bitcoin' in f][0]

btc = pd.read_csv(btc_file)
btc["Date"] = pd.to_datetime(btc["Date"])
btc = btc.sort_values("Date").reset_index(drop=True)
print(f"Bitcoin rows: {len(btc)}")

Bitcoin rows: 2991


In [ ]:
# Feature Engineering 
btc["lag_1"]         = btc["Close"].shift(1)
btc["lag_2"]         = btc["Close"].shift(2)
btc["lag_3"]         = btc["Close"].shift(3)
btc["SMA_7"]         = btc["Close"].rolling(7).mean()
btc["SMA_14"]        = btc["Close"].rolling(14).mean()
btc["EMA_12"]        = btc["Close"].ewm(span=12, adjust=False).mean()
btc["EMA_26"]        = btc["Close"].ewm(span=26, adjust=False).mean()
btc["MACD"]          = btc["EMA_12"] - btc["EMA_26"]
delta                = btc["Close"].diff()
gain                 = delta.clip(lower=0).rolling(14).mean()
loss                 = (-delta.clip(upper=0)).rolling(14).mean()
btc["RSI"]           = 100 - (100 / (1 + gain / (loss + 1e-10)))
btc["log_return"]    = np.log(btc["Close"] / btc["Close"].shift(1))
btc["rolling_std_7"] = btc["Close"].pct_change().rolling(7).std()
btc["momentum_7"]    = btc["Close"] - btc["Close"].shift(7)


# Target = next day log return (teeno models ke liye same)
btc["target"] = btc["log_return"].shift(-1)
btc = btc.dropna().reset_index(drop=True)
print(f"Rows after engineering: {len(btc)}")

Rows after engineering: 2976


In [ ]:
# Split & Scale 
features = [
    "Open", "High", "Low", "Volume",
    "lag_1", "lag_2", "lag_3",
    "SMA_7", "SMA_14", "EMA_12", "EMA_26",
    "MACD", "RSI", "log_return",
    "rolling_std_7", "momentum_7"
]

split = int(len(btc) * 0.8)

X_train = btc[features].iloc[:split]
X_test  = btc[features].iloc[split:]
y_train = btc["target"].iloc[:split].values
y_test  = btc["target"].iloc[split:].values

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

os.makedirs("../models", exist_ok=True)
print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

Train: (2380, 16), Test: (596, 16)


In [ ]:
# Linear Regression
model = LinearRegression()
model.fit(X_train_scaled, y_train)

lr_pred = model.predict(X_test_scaled)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_mae  = mean_absolute_error(y_test, lr_pred)
lr_mda  = np.mean(np.sign(y_test) == np.sign(lr_pred)) * 100

print(f"=== Linear Regression — Bitcoin ===")
print(f"RMSE : {lr_rmse:.6f}")
print(f"MAE  : {lr_mae:.6f}")
print(f"MDA  : {lr_mda:.2f}%")



=== Linear Regression — Bitcoin ===
RMSE : 0.043823
MAE  : 0.028625
MDA  : 50.34%


In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestRegressor

rf_scaler      = StandardScaler()
X_train_rf_sc  = rf_scaler.fit_transform(X_train)
X_test_rf_sc   = rf_scaler.transform(X_test)

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=6,
    min_samples_leaf=10,
    max_features=0.6,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_rf_sc, y_train)

rf_pred = rf_model.predict(X_test_rf_sc)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae  = mean_absolute_error(y_test, rf_pred)
rf_mda  = np.mean(np.sign(y_test) == np.sign(rf_pred)) * 100

print(f"=== Random Forest — Bitcoin ===")
print(f"RMSE : {rf_rmse:.6f}  (LR: {lr_rmse:.6f})")
print(f"MAE  : {rf_mae:.6f}  (LR: {lr_mae:.6f})")
print(f"MDA  : {rf_mda:.2f}%  (LR: {lr_mda:.2f}%)")
print(f"       {'✓ Target MET' if rf_mda >= 55 else '— below 55%'}")



=== Random Forest — Bitcoin ===
RMSE : 0.049036  (LR: 0.043823)
MAE  : 0.033754  (LR: 0.028625)
MDA  : 47.82%  (LR: 50.34%)
       — below 55%


In [ ]:
# LSTM Feature Engineering
def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).ewm(com=period-1, min_periods=period).mean()
    loss  = (-delta.clip(upper=0)).ewm(com=period-1, min_periods=period).mean()
    return 100 - (100 / (1 + gain / loss))

d = btc[['Date','Open','High','Low','Close','Volume']].copy().sort_values('Date').reset_index(drop=True)

d['EMA_12']      = d['Close'].ewm(span=12, adjust=False).mean()
d['EMA_26']      = d['Close'].ewm(span=26, adjust=False).mean()
d['MACD']        = d['EMA_12'] - d['EMA_26']
d['RSI']         = compute_rsi(d['Close'])
bb_mid           = d['Close'].rolling(20).mean()
bb_std           = d['Close'].rolling(20).std()
d['BB_Position'] = (d['Close'] - (bb_mid - 2*bb_std)) / (4*bb_std + 1e-9)
tr               = pd.concat([
                       d['High'] - d['Low'],
                       (d['High'] - d['Close'].shift(1)).abs(),
                       (d['Low']  - d['Close'].shift(1)).abs()
                   ], axis=1).max(axis=1)
d['ATR_14']      = tr.rolling(14).mean()
d['Log_Return']  = np.log(d['Close'] / d['Close'].shift(1))
d['target']      = d['Log_Return'].shift(-1)
d = d.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

LSTM_FEATURES = ['Close', 'Volume', 'RSI', 'MACD', 'BB_Position', 'ATR_14', 'Log_Return']
print(f"LSTM rows: {len(d)}, Features: {len(LSTM_FEATURES)}")

LSTM rows: 2956, Features: 7


In [ ]:
# Split, Scale, Sequences
WINDOW = 75

f_scaler = MinMaxScaler()
X_all    = f_scaler.fit_transform(d[LSTM_FEATURES])
targets  = d['target'].values

def make_sequences(X, y):
    X_seq, y_seq = [], []
    for i in range(WINDOW, len(X)):
        X_seq.append(X[i-WINDOW:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = make_sequences(X_all, targets)
split_seq    = int(len(X_seq) * 0.8)

X_train_s, y_train_s = X_seq[:split_seq], y_seq[:split_seq]
X_test_s,  y_test_s  = X_seq[split_seq:], y_seq[split_seq:]
print(f"Train: {X_train_s.shape}, Test: {X_test_s.shape}")

Train: (2304, 75, 7), Test: (577, 75, 7)


In [ ]:
# Build & Train
import math
from tensorflow.keras.callbacks import LearningRateScheduler

def cosine_schedule(epoch, lr):
    return 1e-6 + 0.5 * (3e-4 - 1e-6) * (1 + math.cos(math.pi * epoch / 100))

lstm_model = Sequential([
    Input(shape=(WINDOW, len(LSTM_FEATURES))),
    LSTM(64, return_sequences=True),  Dropout(0.3),
    LSTM(32, return_sequences=False), Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])
lstm_model.compile(optimizer=Adam(3e-4), loss='mse')
lstm_model.fit(
    X_train_s, y_train_s,
    epochs=100, batch_size=16,
    validation_data=(X_test_s, y_test_s),
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=20,
                      restore_best_weights=True, verbose=1),
        LearningRateScheduler(cosine_schedule, verbose=0)
    ],
    verbose=1
)
lstm_model.save("models/btc_lstm.keras")

Epoch 1/100



  1/144 ━━━━━━━━━━━━━━━━━━━━ 1:36 675ms/step - loss: 0.0043


  7/144 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0034   


 13/144 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0033 


 19/144 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0032

...


139/144 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0018


144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0018


144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0018 - val_loss: 0.0018 - learning_rate: 2.6863e-04


Epoch 22: early stopping


Restoring model weights from the end of the best epoch: 2.


In [ ]:
# Evaluation
preds_log  = lstm_model.predict(X_test_s, verbose=0).flatten()
actual_log = y_test_s

lstm_rmse = np.sqrt(mean_squared_error(actual_log, preds_log))
lstm_mae  = mean_absolute_error(actual_log, preds_log)
lstm_mda  = np.mean(np.sign(actual_log) == np.sign(preds_log)) * 100

print(f"=== LSTM — Bitcoin (Log Return) ===")
print(f"RMSE : {lstm_rmse:.6f}  (RF: {rf_rmse:.6f}  |  LR: {lr_rmse:.6f})")
print(f"MAE  : {lstm_mae:.6f}  (RF: {rf_mae:.6f}  |  LR: {lr_mae:.6f})")
print(f"MDA  : {lstm_mda:.2f}%  (RF: {rf_mda:.2f}%  |  LR: {lr_mda:.2f}%)")

=== LSTM — Bitcoin (Log Return) ===
RMSE : 0.042422  (RF: 0.049036  |  LR: 0.043823)
MAE  : 0.026986  (RF: 0.033754  |  LR: 0.028625)
MDA  : 53.90%  (RF: 47.82%  |  LR: 50.34%)
